# Data Exploration

In [ ]:
from pathlib import Path
import pandas as pd

Before develing into the individual players, I wanted to see how Eagles performed as a team relative to the rest of the league

In [ ]:
RAW_DIR = Path("..") / "data" / "raw"
team_stats = pd.read_parquet(RAW_DIR / "team_stats__2007.parquet").query("season_type == 'REG'")
schedules = pd.read_parquet(RAW_DIR / "schedules__2007.parquet").query("game_type == 'REG'")

In [ ]:
team_stats.head()

In [ ]:
schedules

When looking at this I noticed that they called an opponent "LA". However, there was no NFL team in LA and in fact the game idea correctly states that this was a game against the Saint Louis Rams. This shows a discrepency between how the team_stats df and the schedules df are expressing things. So I'm going to fix that by using the acronyms from 2007.

In [ ]:
TEAM_FIX = {
    "LA": "STL",
    "LAC": "SD",
    "LV": "OAK",
}

team_stats[['team', 'opponent_team']] = team_stats[['team', 'opponent_team']].replace(TEAM_FIX)
team_stats

In [ ]:
cols = ["game_id", "team", "pts_for", "pts_against"]


Get the home and away stats separated by teams to then seperate. `set_axis` is used to fill out the cols values with the schedules data from the schedules, Then points can be created by concatanating the home and away games. 

In [ ]:
home = schedules[["game_id", "home_team", "home_score", "away_score"]].set_axis(cols, axis=1)
away = schedules[["game_id", "away_team", "away_score", "home_score"]].set_axis(cols, axis=1)
points = pd.concat([home, away], ignore_index=True)
points


Now we will find the offense stats per game, so that later we can seperate it and easily gets offense and defense for the season and a per game basis

In [ ]:
offense = pd.DataFrame({
    "game_id": team_stats["game_id"],
    "team": team_stats["team"],
    "opponent": team_stats["opponent_team"],
    "pass_att": team_stats["attempts"], #does not include sacks
    "sacks": team_stats["sacks_suffered"], # how many times the offense was sacked
    "dropbacks": team_stats["attempts"] + team_stats["sacks_suffered"], # How many times a passing play was made
    "pass_yards": team_stats["passing_yards"],
    "sack_yards": team_stats["sack_yards_lost"],
    "net_pass_yards": team_stats["passing_yards"] + team_stats["sack_yards_lost"], # how many yards on a passing play (including the negatives)
    "pass_tds": team_stats["passing_tds"],
    "pass_epa": team_stats["passing_epa"],
    "rush_att": team_stats["carries"],
    "rush_yards": team_stats["rushing_yards"],
    "rush_epa": team_stats["rushing_epa"],
    "ints": team_stats["passing_interceptions"],
    "fumbles": team_stats["fumbles_total"], #number of times fumbled
    "fumbles_lost": team_stats["fumbles_lost_total"]
})

offense["total_yards"] = offense["net_pass_yards"] + offense["rush_yards"]
offense["total_epa"] = offense["pass_epa"] + offense["rush_epa"]
offense["total_turnovers"] = offense["ints"] + offense["fumbles_lost"]
offense

Note. Especially with someone like McNabb I think its worth looking at a **true** play-by-play of passing plays vs rushing plays. So scrambles included in passing plays. I'd need pbp for this though. 